# Brain Tumor Segmentation — Corrected Reproducible Workflow

> **Research-only / not for diagnosis.** This notebook runs the refactored project scripts. The original archive did **not** contain the MRI images or annotation file. Use only data you are authorized to process.

**Models:** compact U-Net, DeepLabV3+ (MobileNetV2), or ResNet50 U-Net. **Metrics:** Dice, IoU, and pixel accuracy on val/test splits. **Important:** image-based splits are not proof of patient-level generalization.

## 1. Configure paths

Install dependencies from the repository root first: `python -m pip install -r requirements.txt`. Start Jupyter from the repository root or the `notebooks` directory.

In [ ]:
from pathlib import Path
import sys
import json
import subprocess

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Open this notebook from the repository root or notebooks directory")

IMAGES = ROOT / "data/raw/datas/all"
ANNOTATIONS = ROOT / "data/raw/datas/annotations_all.json"
MASKS = ROOT / "data/processed/masks"
MODEL_NAME = "unet"    # change to "deeplabv3plus" or "resnet50_unet"
RUN_DIR = ROOT / "runs" / MODEL_NAME
EPOCHS = 30
GROUP_REGEX = None    # e.g. r"^(patient_\d+)_" IF filenames truly encode patient IDs
print("Repository:", ROOT)
print("Images available:", IMAGES.exists())
print("Annotations available:", ANNOTATIONS.exists())

## 2. Prepare *lossless* binary masks

The original notebook saved masks as JPEG files. Here, polygons are rasterized into `*.png` with exact 0/255 values; categorical masks are resized using nearest-neighbor interpolation.

In [ ]:
if not IMAGES.is_dir() or not ANNOTATIONS.is_file():
    raise FileNotFoundError("Place your authorized original scans and annotations in data/raw/datas/ first")
subprocess.run([sys.executable, str(ROOT / "scripts/prepare_masks.py"),
                "--images", str(IMAGES), "--annotations", str(ANNOTATIONS),
                "--output", str(MASKS)], check=True, cwd=ROOT)

## 3. Train a model, preserving a **held-out test set**

By default 70% of images are training data, 15% are used for validation/early stopping, and 15% remain held out for final evaluation. If patient identifiers exist, set `GROUP_REGEX` to create group-disjoint splits.

In [ ]:
command = [sys.executable, str(ROOT / "scripts/train.py"),
           "--images", str(IMAGES), "--masks", str(MASKS),
           "--model", MODEL_NAME, "--out", str(RUN_DIR),
           "--epochs", str(EPOCHS)]
if GROUP_REGEX:
    command += ["--group-regex", GROUP_REGEX]
subprocess.run(command, check=True, cwd=ROOT)

## 4. Read saved metrics

No new performance numbers are claimed until these cells run on the actual dataset. A high pixel accuracy alone can be misleading if the tumor foreground occupies a small part of each scan.

In [ ]:
metrics = json.loads((RUN_DIR / "metrics.json").read_text())
print(json.dumps(metrics, indent=2))
print("Split lists:", RUN_DIR / "splits.json")

## 5. Plot a test example (model + ground-truth mask)

This visualization uses the selected checkpoint, rather than an unrelated model variable. Do not use the held-out test examples for choosing hyperparameters.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from brain_tumor_segmentation.data import list_pairs, load_arrays
from brain_tumor_segmentation.models import preprocess

splits = json.loads((RUN_DIR / "splits.json").read_text())
pairs = {image.name: (image, mask) for image, mask in list_pairs(IMAGES, MASKS)}
test_example = [pairs[splits["test"][0]]]
images, truth = load_arrays(test_example, 256)
model = tf.keras.models.load_model(RUN_DIR / "best.keras", compile=False)
prob = model.predict(preprocess(images, MODEL_NAME), verbose=0)[0, :, :, 0]
mask = (prob >= 0.5).astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(images[0]); axes[0].set_title("Input")
axes[1].imshow(truth[0, :, :, 0], cmap="gray", vmin=0, vmax=1); axes[1].set_title("Reference mask")
axes[2].imshow(mask, cmap="gray", vmin=0, vmax=1); axes[2].set_title("Model prediction")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

## 6. Optional: explore another model or local demo

Change `MODEL_NAME` above and rerun to compare architectures. Each run writes to its own folder. Run `python scripts/app.py` from the repository root for the **supplied historical U-Net** Gradio demo. For your newly trained model, run `python scripts/app.py --model runs/unet/best.keras --metadata runs/unet/metadata.json`.

**Provenance:** This notebook is a clean, runnable replacement for the supplied Colab notebook. See `docs/METHODOLOGY.md` for corrections and differences from the original experiments.